In [18]:
from langgraph.graph import START,END,StateGraph
from typing import TypedDict,Annotated
from langchain_core.messages import HumanMessage,BaseMessage
from langgraph.graph import add_messages
from langchain_groq import ChatGroq
from dotenv import load_dotenv

from langgraph.prebuilt import ToolNode,tools_condition
from langchain_community.tools import DuckDuckGoSearchRun # ye prebuilt tool hai (search engine) lang chain ka 
from langchain_core.tools import tool  

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings


In [5]:
load_dotenv()

True

In [6]:
model = ChatGroq(
    model = "llama-3.1-8b-instant"
)

In [8]:
# RAG - step 1 (load the document)
loader = PyPDFLoader('Agentic AI.pdf')
docs = loader.load()

In [9]:
len(docs)

53

In [12]:
## RAG - step 2 (split the document)
splitter = RecursiveCharacterTextSplitter(chunk_size = 100, chunk_overlap = 50) 
# chunk size ----- hrr chunk ka size kitnba ho ga 
# chunk overlap ----- taa k chunks k drmyan context ko retain kya ja ske

chunks = splitter.split_documents(docs) 


In [13]:
len(chunks)

53

In [21]:
## RAG - step 3 (Generate embeddings and store in vectore database)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
vector_store = FAISS.from_documents(chunks,embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9363.10it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [25]:
vector_store

In [24]:
##  RAG - step 4 (Retriever)
retriever = vector_store.as_retriever(search_type = 'similarity',search_kwargs={'k':4})

In [26]:
## RAG as a tool
@tool
def rag_tool(query):
    """ Retrieve the relevant information from pdf"""
    result = retriever.invoke(query)

    context = [doc.page_content for doc in result] # jo embeddings ... retriever ne retrieve krni hain vector store se jo as a context model k pass jaye ga 
    metadata = [doc.metadata for doc in result]

    return {
        "query": query,
        "context": context,
        "metadata": metadata
    }

In [27]:
tools = [rag_tool]

In [28]:
# MAke LLM aware of tools
llm_with_tools = model.bind_tools(tools)

In [29]:
# Creating State
class chatState(TypedDict):
    messages:Annotated[list[BaseMessage],add_messages]


In [30]:
# Chat Node 
def chat_node(state: chatState):
    messages = state['messages']
    response= llm_with_tools.invoke(messages)
    return {"messages": [response]}

# Tool NOde
tool_node = ToolNode(tools) # Execute the tools

In [32]:
# Graph Structure

graph = StateGraph(chatState)

graph.add_node("chat_node",chat_node)
graph.add_node("tools",tool_node)

graph.add_edge(START,"chat_node")
graph.add_conditional_edges("chat_node",tools_condition) # ye tools_condition ne decide krna hai k kon sa node execute ho ga (tool node) ya direct answer aaye ga 
graph.add_edge("tools","chat_node")

workflow = graph.compile()

In [36]:
import traceback

try:
    out = workflow.invoke({
        "messages": [HumanMessage(content="Using the notes, Explain what is Agentic AI?")]
    })
    print(out["messages"][-1].content)
except Exception as e:
    traceback.print_exc()

Based on the output, Agentic AI refers to a type of artificial intelligence that has the ability to act on its own in a way that is intentional and goal-directed. It is characterized by its ability to take initiative, make decisions, and adapt to changing situations. Agentic AI is often associated with the idea of autonomy and self-directed behavior.

Agentic AI has the potential to revolutionize various fields, including robotics, healthcare, finance, and education. It can be used to develop more sophisticated and autonomous systems that can learn from experience, make decisions, and interact with humans in a more natural and intuitive way.

However, the development and deployment of Agentic AI also raise several concerns, including issues related to safety, security, and accountability. As Agentic AI systems become more advanced and autonomous, it is essential to ensure that they are designed and developed in a way that prioritizes human values and well-being.

Overall, Agentic AI is